# PhenoAssistant Case 1 on Microsoft Agent Framework

CPU-only migration of the analytical portion of `case1.ipynb`.

This notebook reuses the tracked phenotype table `results/Case1/aracrop_phenotypes.csv` and routes the migrated analysis tasks through the production MAF application. The original AutoGen notebook remains unchanged.

The current CPU scope migrates longitudinal plotting, ecotype ranking, and the repeated-measures ANOVA + Tukey workflow. Image segmentation, phenotype extraction from source images, reusable pipeline execution, and RAG validation remain explicit later-phase boundaries rather than being simulated here.

In [ ]:
from __future__ import annotations

import inspect
import os
from pathlib import Path
from typing import Any

from phenoassistant_maf import (
    OpenRouterSettings,
    build_application,
    create_chat_client,
    run_application,
)

ROOT = Path.cwd()
CASE1_DATA = ROOT / "results/Case1/aracrop_phenotypes.csv"
BASE_DATA = ROOT / "results/demo/potato_phenotypes.csv"
OUTPUT_DIR = ROOT / "results/maf_case1"

assert CASE1_DATA.is_file()
assert BASE_DATA.is_file()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CASE1_ROWS=", sum(1 for _ in CASE1_DATA.open()) - 1)
print("CASE1_DATA=", CASE1_DATA.relative_to(ROOT))


## Original Tasks 1 / 1.1 / 1.2 / 1.3 — deferred boundary

The original Case 1 notebook first performs instance segmentation and phenotype extraction for 24 Arabidopsis plants, then saves, lists, and re-executes an `ara_crop_pipeline`.

Those operations are intentionally **not** reproduced by loading legacy `extracted_pipelines.py` or by exposing arbitrary pipeline code execution to the model. The current notebook resumes from the tracked `aracrop_phenotypes.csv` artifact. Image inference and bounded/allowlisted pipeline execution belong to the GPU/pipeline migration phase.

In [ ]:
if not os.environ.get("OPENROUTER_API_KEY"):
    raise RuntimeError("Set OPENROUTER_API_KEY before running this cell.")

os.environ.setdefault("OPENROUTER_MODEL", "openrouter/free")

settings = OpenRouterSettings.from_env()
client = create_chat_client(settings)


def forbidden_calculator(a: int, b: int, operator: str) -> int:
    raise AssertionError("Case 1 selected calculator unexpectedly.")


def forbidden_anova(
    data_path: str,
    descriptor: str,
    within_subject_factor: str,
    between_subject_factor: str,
    subject_id: str,
    save_path: str | None = None,
) -> list[dict[str, Any]]:
    raise AssertionError("Case 1 selected the generic ANOVA tool unexpectedly.")


def forbidden_tukey(
    data_path: str,
    descriptor: str,
    between_subject_factor: str,
    subject_id: str,
    save_path: str | None = None,
) -> list[dict[str, Any]]:
    raise AssertionError("Case 1 selected the generic Tukey tool unexpectedly.")


def forbidden_regression(
    data_path: str,
    x_column: str,
    y_column: str,
    plot_path: str,
) -> dict[str, float]:
    raise AssertionError("Case 1 selected regression unexpectedly.")


def forbidden_aggregate(
    data_path: str,
    operation: str,
    value_column: str,
    filter_column: str,
    filter_value: str,
) -> dict[str, Any]:
    raise AssertionError("Case 1 selected CSV aggregation unexpectedly.")


application = build_application(
    client=client,
    data_path=str(BASE_DATA),
    calculator_callable=forbidden_calculator,
    anova_callable=forbidden_anova,
    tukey_callable=forbidden_tukey,
    regression_callable=forbidden_regression,
    aggregate_callable=forbidden_aggregate,
    case1_data_path=str(CASE1_DATA),
    case1_output_dir=str(OUTPUT_DIR),
)

print("MODEL=", settings.model)
print("TOOLS=", application.registry.names)


In [ ]:
def response_text(response: Any) -> str:
    messages = getattr(response, "messages", None)
    if messages:
        text = getattr(messages[-1], "text", None)
        if isinstance(text, str):
            return text.strip()
    return str(response).strip()


In [ ]:
# Original Task 2: longitudinal phenotype statistics and visualisation.
plot_response = await run_application(
    application,
    '''
    Using the trusted Case 1 Arabidopsis phenotype dataset, call
    plot_longitudinal_phenotypes exactly once for leaf_count,
    projected_leaf_area, diameter, and perimeter. Summarise only the
    returned evidence, including which plots and statistics files were
    produced.
    ''',
)
print(response_text(plot_response))


In [ ]:
# Original Task 3: rank ecotypes by PLA.
# The migrated tool ranks from trusted phenotype data rather than visually
# inferring values from pla_plot.png.
ranking_response = await run_application(
    application,
    '''
    Using the trusted Case 1 data, call rank_ecotypes_by_phenotype
    exactly once with projected_leaf_area. Report the ecotypes from
    largest to smallest PLA using only returned tool evidence.
    ''',
)
print(response_text(ranking_response))


In [ ]:
# Original Task 4: mixed-design repeated-measures ANOVA + Tukey-Kramer.
statistics_response = await run_application(
    application,
    '''
    For projected_leaf_area in the trusted Case 1 dataset, call
    analyse_repeated_measures_with_posthoc exactly once. Report whether
    the time-ecotype interaction is significant at P < 0.01, summarise
    the Tukey-Kramer evidence using the tool's P < 0.05 post-hoc
    threshold, and report any evidence-supported large, medium, and
    small PLA groups. Use only returned tool evidence.
    ''',
)
print(response_text(statistics_response))


## Original Task 5 — RAG / Phenotiki comparison deferred

The original notebook asks the agent to compare the Case 1 findings with the Phenotiki paper. No production RAG/retrieval tool is registered in the current MAF profile, so this notebook does not fabricate that capability.

The preserved Case 1 reference result is: a significant time–ecotype interaction for PLA, with large PLA (`ein2`, `col0`), medium PLA (`adh1`, `pgm`), and small PLA (`ctr`). The migrated analytical tools independently reproduce those values from the tracked dataset; literature-grounded retrieval remains a separate migration task.

In [ ]:
expected_outputs = (
    OUTPUT_DIR / "leaf_count_plot.png",
    OUTPUT_DIR / "leaf_count_stats.csv",
    OUTPUT_DIR / "pla_plot.png",
    OUTPUT_DIR / "pla_stats.csv",
    OUTPUT_DIR / "plant_diameter_plot.png",
    OUTPUT_DIR / "plant_diameter_stats.csv",
    OUTPUT_DIR / "plant_perimeter_plot.png",
    OUTPUT_DIR / "plant_perimeter_stats.csv",
)

for path in expected_outputs:
    assert path.is_file()
    assert path.stat().st_size > 0

print("CASE1_MAF_CPU_WORKFLOW_COMPLETE")


async def close_provider(value: Any) -> None:
    for candidate in (
        value,
        getattr(value, "client", None),
        getattr(value, "_client", None),
        getattr(value, "_openai_client", None),
    ):
        if candidate is None:
            continue
        for name in ("aclose", "close"):
            method = getattr(candidate, name, None)
            if callable(method):
                result = method()
                if inspect.isawaitable(result):
                    await result
                return


await close_provider(client)
print("OPENROUTER_CLIENT_CLOSED")
